In [1]:
# =========================================================
# 02_preprocess.ipynb | Full Preprocessing (Clean Code)
# =========================================================
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ---------------------------------------------------------
# 1) Load Data
# ---------------------------------------------------------
ROOT = Path().resolve().parent   # یک پوشه بالاتر از notebooks
data_path = ROOT / "data" / "raw" / "Housing.csv"
df = pd.read_csv(data_path)

# ---------------------------------------------------------
# 2) Target & Features
# ---------------------------------------------------------
TARGET = "price"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# ---------------------------------------------------------
# 3) Identify Categorical & Numeric Columns
# ---------------------------------------------------------
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# ---------------------------------------------------------
# 4) Missing Values Check
# ---------------------------------------------------------
missing_summary = df.isna().sum()
print("Missing values per column:\n", missing_summary)

# ---------------------------------------------------------
# 5) Preprocessing Pipelines
# ---------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# ---------------------------------------------------------
# 6) Train/Test Split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 7) Fit-Transform Example (Preprocessing Only)
# ---------------------------------------------------------
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train shape (processed):", X_train_processed.shape)
print("Test shape (processed):", X_test_processed.shape)


Missing values per column:
 price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64
Train shape (processed): (436, 13)
Test shape (processed): (109, 13)


In [2]:
import pandas as pd
import numpy as np
from scipy import sparse
from pathlib import Path

# مسیر ذخیره
ROOT = Path().resolve().parent   # یک پوشه بالاتر از notebooks
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# نام ستون‌ها
feature_names = preprocessor.get_feature_names_out()

def to_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# تبدیل به DataFrame
X_train_df = pd.DataFrame(to_dense(X_train_processed), columns=feature_names)
X_test_df  = pd.DataFrame(to_dense(X_test_processed), columns=feature_names)

# ذخیره CSV
X_train_df.to_csv(PROCESSED_DIR / "X_train_processed.csv", index=False)
X_test_df.to_csv(PROCESSED_DIR / "X_test_processed.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

print("✅ Processed CSVs saved successfully.")


✅ Processed CSVs saved successfully.
